# S16 – Zeta potential verified against the publications, and separated by measurement environment; surface charge class from its sign

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
from S16_comments import Audit, apply_comments   # the code of the comments of the step, apart from the data handling

INPUT = "../../S15_Sphericity_not_confirmed/S15.xlsx"
OUTPUT = "../S16.xlsx"
COMMENTS = "../S16_comments.xlsx"
AUDIT = "../S16_audit.xlsx"
STEP = "S16"
GUL, LABOUTA, NANOPHAROS = "Gul. Et al..", "Labouta Et. Al. ", "Nanopharos"

df = pd.read_excel(INPUT, keep_default_na=False, na_values=[""]).astype(object)
print(len(df), "records before the step")
audit = Audit(STEP, df, before=INPUT)
put, drop = audit.put, audit.drop                  # a rule sets a value / removes records through the audit, which records the change
changes, removed = audit.changes, audit.removed    # the ledger of the step

3043 records before the step


### 1. Values from the publication review: the comments of this step

In [2]:
# every value from the publication review: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S16_curated_L1_doi', 'S16_curated_L3_coat', 'S16_curated_L4_diameter', 'S16_curated_L7_row'])

318 comments applied (S16_curated_L1_doi, S16_curated_L3_coat, S16_curated_L4_diameter, S16_curated_L7_row): 318 values set, 0 records removed


### 2. Corrections from the check against the publications: the comments of this step

In [3]:
# every correction from the check of this field against the publications: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S16_publication_check_01', 'S16_publication_check_02', 'S16_publication_check_03', 'S16_publication_check_04', 'S16_publication_check_05', 'S16_publication_check_06', 'S16_publication_check_07', 'S16_publication_check_08', 'S16_publication_check_09', 'S16_publication_check_10', 'S16_publication_check_11', 'S16_publication_check_12', 'S16_publication_check_13', 'S16_publication_check_14', 'S16_publication_check_15', 'S16_publication_check_16', 'S16_publication_check_17', 'S16_publication_check_18'])

782 comments applied (S16_publication_check_01, S16_publication_check_02, S16_publication_check_03, S16_publication_check_04, S16_publication_check_05, S16_publication_check_06, S16_publication_check_07, S16_publication_check_08, S16_publication_check_09, S16_publication_check_10, S16_publication_check_11, S16_publication_check_12, S16_publication_check_13, S16_publication_check_14, S16_publication_check_15, S16_publication_check_16, S16_publication_check_17, S16_publication_check_18): 782 values set, 0 records removed


### 3. Charge class from the sign of the verified zeta potential (in water; without a value in water, the reported value for the sources without a charge column); a note where the class comes from the source file

In [4]:
CHARGE_NOTE = ' Surface_Charge as given in the source file (no zeta potential value in the dataset).'
from_pub = {c["raw_id"] for c in changes if c["column"] == "Surface_Charge"}   # classes set above from the publication review and the corrections
for i in df.index:
    sc, z, rep = df.at[i, "Surface_Charge"], df.at[i, "Zeta_in_Water (mV)"], df.at[i, "Zeta potential (mV)"]
    if pd.isna(sc) and not pd.isna(z):
        put(i, "Surface_Charge", "Positive" if float(z) > 0 else ("Negative" if float(z) < 0 else "Neutral"), f"{STEP}_charge_from_zeta",
            "surface charge class = sign of the zeta potential in water (class missing in the source)")
    elif pd.isna(sc) and df.at[i, "_src"] != GUL and not pd.isna(rep):
        put(i, "Surface_Charge", "Positive" if float(rep) > 0 else ("Negative" if float(rep) < 0 else "Neutral"), f"{STEP}_charge_from_zeta",
            "surface charge class = sign of the reported zeta potential (no value in water; source has no charge column)")
    elif (not pd.isna(sc) and all(pd.isna(df.at[i, c]) for c in ["Zeta_in_Water (mV)", "Zeta_in_Medium (mV)", "Zeta_in_Other (mV)"])
          and df.at[i, "raw_id"] not in from_pub):
        com = "" if pd.isna(df.at[i, "Size_Measurement_Comments"]) else str(df.at[i, "Size_Measurement_Comments"])
        if "Surface_Charge" not in com:   # no note when the comment already states the charge or its source (never doubled)
            put(i, "Size_Measurement_Comments", (com.rstrip() + CHARGE_NOTE).strip(), f"{STEP}_charge_source_note",
                "the charge class comes from the source file; no zeta potential value in any column")

### The table after the step and its audit

In [5]:
# the table after the step; the audit module compares it with the table before the step and writes the audit workbook
df.to_excel(OUTPUT, index=False)
audit.write(AUDIT)

0 records removed and 1750 cells changed compared with the table before the step


3043 records after the step | 0 records removed | 1750 cells changed | ../S16_audit.xlsx
